## Example 1: Sign a request with SigV4

In [ ]:
import importlib.metadata

from aws_sdk_signers import URI, AWSCredentialIdentity, AWSRequest, Field, Fields, SigV4Signer

# Example 1: the signer adds Authorization and X-Amz-Date to an HTTP request description
assert importlib.metadata.version("aws-sdk-signers") == "0.4.0"
request = AWSRequest(destination=URI(host="bedrock-runtime.us-east-1.amazonaws.com", path="/model/m/converse"), method="POST",
                     body=[b'{"messages": []}'], fields=Fields([Field(name="Content-Type", values=["application/json"])]))
identity = AWSCredentialIdentity(access_key_id="AKIDEXAMPLE", secret_access_key="secret")
signed = SigV4Signer().sign(request=request, identity=identity, properties={"region": "us-east-1", "service": "bedrock"})
auth = signed.fields["Authorization"].as_string()
assert auth.startswith("AWS4-HMAC-SHA256 Credential=AKIDEXAMPLE/") and "/us-east-1/bedrock/aws4_request" in auth, auth
assert "X-Amz-Date" in signed.fields or "x-amz-date" in signed.fields
print("Example 1 OK:", auth[:100])

## Example 2: A fixed date gives a reproducible signature

In [ ]:
from aws_sdk_signers import URI, AWSCredentialIdentity, AWSRequest, Fields, SigV4Signer

def sign(body):
    req = AWSRequest(destination=URI(host="example.amazonaws.com", path="/"), method="GET", body=[body], fields=Fields())
    out = SigV4Signer().sign(request=req, identity=AWSCredentialIdentity(access_key_id="AK", secret_access_key="SK"),
                             properties={"region": "us-east-1", "service": "svc", "date": "20260102T030405Z"})
    return out.fields["Authorization"].as_string()

assert sign(b"") == sign(b"") and sign(b"") != sign(b"x")
assert "Credential=AK/20260102/us-east-1/svc/aws4_request" in sign(b"")
print("Example 2 OK:", sign(b"").split("Signature=")[1][:16])

## Example 3: The signature matches botocore's for the same request

In [ ]:
import datetime
from unittest import mock

from aws_sdk_signers import URI, AWSCredentialIdentity, AWSRequest, Field, Fields, SigV4Signer
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest as BotoRequest
from botocore.credentials import Credentials

# Example 3: two independent SigV4 implementations agree byte for byte
date = "20260102T030405Z"
ours = SigV4Signer().sign(
    request=AWSRequest(destination=URI(host="h.amazonaws.com", path="/p", query="x=1"), method="GET", body=None,
                       fields=Fields([Field(name="X-Amz-Date", values=[date])])),
    identity=AWSCredentialIdentity(access_key_id="AK", secret_access_key="SK"), properties={"region": "us-east-1", "service": "svc", "date": date})
boto = BotoRequest(method="GET", url="https://h.amazonaws.com/p?x=1", data=b"")
with mock.patch("botocore.auth.get_current_datetime", return_value=datetime.datetime(2026, 1, 2, 3, 4, 5)):
    SigV4Auth(Credentials("AK", "SK"), "svc", "us-east-1").add_auth(boto)
sig = lambda h: h.split("Signature=")[1]
assert sig(ours.fields["Authorization"].as_string()) == sig(boto.headers["Authorization"]), (ours.fields["Authorization"].as_string(), boto.headers["Authorization"])
print("Example 3 OK: same signature", sig(boto.headers["Authorization"])[:16])

## Example 4: Session tokens are signed

In [ ]:
from aws_sdk_signers import URI, AWSCredentialIdentity, AWSRequest, Fields, SigV4Signer

# Example 4: temporary credentials add X-Amz-Security-Token and include it in the signed headers
req = AWSRequest(destination=URI(host="example.amazonaws.com", path="/"), method="GET", body=None, fields=Fields())
signed = SigV4Signer().sign(request=req, identity=AWSCredentialIdentity(access_key_id="ASIA", secret_access_key="SK", session_token="tok"),
                            properties={"region": "eu-west-1", "service": "s3"})
names = {f.name.lower() for f in signed.fields}
assert "x-amz-security-token" in names, names
assert "x-amz-security-token" in signed.fields["Authorization"].as_string()
print("Example 4 OK:", sorted(names))

## Example 5: The async signer gives the same result

In [ ]:
from aws_sdk_signers import URI, AsyncSigV4Signer, AWSCredentialIdentity, AWSRequest, Fields, SigV4Signer

# Example 5: AsyncSigV4Signer is the asyncio twin used by the new SDK's clients
props = {"region": "us-west-2", "service": "svc", "date": "20260102T030405Z"}
ident = AWSCredentialIdentity(access_key_id="AK", secret_access_key="SK")
make = lambda: AWSRequest(destination=URI(host="example.amazonaws.com", path="/"), method="GET", body=None, fields=Fields())
sync = SigV4Signer().sign(request=make(), identity=ident, properties=props).fields["Authorization"].as_string()
asynchronous = (await AsyncSigV4Signer().sign(request=make(), identity=ident, properties=props)).fields["Authorization"].as_string()
assert sync == asynchronous, (sync, asynchronous)
print("Example 5 OK")